<a href="https://colab.research.google.com/github/Danial-071/langchain-project/blob/main/langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install langchain-nvidia-ai-endpoints
!pip install pypdf
!pip install lark
!pip install posthog
!pip install chromadb
!pip install langchain-classic -q
!pip install langchain-community -q
!pip install databricks_langchain -q

In [ ]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
print("Everything set!")

Everything set!


In [ ]:
def llm():
    from langchain_openrouter import ChatOpenRouter
    model = ChatOpenRouter(model = 'nvidia/nemotron-3-ultra-550b-a55b:free')
    return model


In [ ]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

def text_splitter(data, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

In [ ]:
from langchain_classic.embeddings import HuggingFaceEmbeddings

def embeddings():
    embeddings = HuggingFaceEmbeddings(model_name ='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader('./companypolicies.txt')
data = loader.load()

In [ ]:
chunks_txt = text_splitter(data, 200, 20)

In [ ]:
from langchain_classic.vectorstores import Chroma

db = Chroma.from_documents(chunks_txt, embeddings())

In [ ]:
retriever = db.as_retriever(search_type="similarity_score_threshold", search_kwargs={"k": 90,'score_threshold': 0.8})

retriever.invoke('smoking policy')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()

In [ ]:
pdf_data[4]

In [ ]:
# Split
chunks_pdf = text_splitter(pdf_data, 500, 20)

ids = db.get()["ids"]
db.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.
db = Chroma.from_documents(documents=chunks_pdf, embedding=embeddings())

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever = db.as_retriever(), llm=llm()
)

In [ ]:
docs = retriever.invoke(query)
docs

In [ ]:
from langchain_core.documents import Document

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever


In [ ]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]
type(docs[0])

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",  )
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

In [ ]:
db = Chroma.from_documents(docs, embeddings())

In [ ]:
document_content_description = "Brief summary of a movie."

from databricks_langchain import DatabricksVectorSearch

retriever = SelfQueryRetriever.from_llm(
    llm(),
    db,
    document_content_description,
    metadata_field_info,
)

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.storage import InMemoryStore

In [ ]:

parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [ ]:

vectordb = Chroma.from_documents(
    documents = docs, embedding = embeddings()
)

store = InMemoryStore()

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter = child_splitter,
    parent_splitter = parent_splitter
)

In [ ]:
retriever.add_documents(docs)

In [ ]:
len(list(store.yield_keys()))

In [ ]:
sub_docs = vectordb.similarity_search("smoking policy")

In [ ]:
print(sub_docs[0].metadata)

{'year': 2006, 'rating': 8.6, 'director': 'Satoshi Kon'}


In [ ]:
retrieved_docs = retriever.invoke("smoking policy")
print(retrieved_docs[0].page_content)

A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea


In [ ]:
vectordb = Chroma.from_documents(docs, embeddings())

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

# This example specifies a query with filter
retriever.invoke(
    "I want to watch a movie directed by Greta Gerwig"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Document(metadata={'rating': 8.3, 'year': 2019, 'director': 'Greta Gerwig'}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(metadata={'rating': 8.3, 'doc_id': '27639dae-f2db-425d-be36-9a233431827a', 'year': 2019, 'director': 'Greta Gerwig'}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them')]